# Milestone 2: frozen quartet transfer

Evaluate the final quartet-fit checkpoint on **seven other complete arrangements**
from the training-source corpus. Each has the same balanced 576 images / 1,728 questions.
Recheck the learned arrangement against its saved predictions: **13,824 questions total**.

This is descriptive transfer evaluation with no training, accuracy gates, or reserved
validation/test inference. Arrangements are unseen by this checkpoint, not an untouched
project test split. Keep R=2, float32, batch 32 and the original rendering.

Enable GPU and internet; attach `milestone2_quartet_fit_artifacts.zip` and set a committed
`REPO_REF`. See `docs/milestones/milestone2_quartet_transfer.md`.


In [ ]:
REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
REPO_REF = "milestone2"  # Commit SHA preferred; must contain this implementation.
REPO_DIR = "/kaggle/working/multi-modal-loop-quartet-transfer"
RUN_ROOT = "/kaggle/working/milestone2_quartet_transfer"
REFERENCE_SOURCE = (
    "/kaggle/input/REPLACE_WITH_YOUR_ARTIFACT_PATH/milestone2_quartet_fit_artifacts.zip"
)


## Checkout and install

Run all cells in order. The resolved revision is recorded with the artifacts.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "to resume, or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## Audit and select populations

Verify the pinned checkpoint, corpus and saved predictions. Enumerate every complete
balanced arrangement, excluding learned origins from the transfer populations. Preserve
family and question matching without selecting on performance. ZIPs and extracted run
directories are accepted. Use fresh output paths.


In [ ]:
import multimodal_loop.eval.kaggle_quartet_transfer as helpers
from multimodal_loop.eval.kaggle_quartet_transfer import (
    archive_quartet_transfer,
    prepare_quartet_transfer,
    run_quartet_transfer,
)

if not Path(helpers.__file__).resolve().is_relative_to(REPO_DIR / "src"):
    raise RuntimeError("Another package checkout is cached; restart the kernel")
run = prepare_quartet_transfer(REPO_DIR, RUN_ROOT, REFERENCE_SOURCE)


## Frozen evaluation

Run the learned reproduction control and all seven transfer populations on the same
frozen final checkpoint. No optimizer is created and no checkpoint is selected or updated.


In [ ]:
report = run_quartet_transfer(run)


## Inspect reproduction and transfer separately

Any reproduction disagreement is prominently flagged. Preserve the archive even if
reproduction fails; investigate the discrepancy before interpreting transfer scores.
The transfer aggregate excludes the learned arrangement. No transfer pass/fail gate applies.


In [ ]:
import json

from IPython.display import HTML, FileLink, display

print("Reproduction:", report["reproduction"])
if not report["reproduction"]["passed"]:
    print("REPRODUCTION FAILED: investigate reference disagreement before interpreting transfer.")
for name, metrics in report["arrangements"].items():
    print(name, {k: metrics["summary"][k] for k in ("total", "accuracy", "loss")})
    print("Per shape:", metrics["summary"]["breakdowns"]["shape"])
    keys = ("total", "correct", "fraction")
    print("Correct families:", {k: metrics["families"][k] for k in keys})
aggregate = report["transfer_aggregate"]
print("Transfer only:", {k: aggregate["summary"][k] for k in ("total", "accuracy", "loss")})
print("Matched changes:", json.dumps(report["matched_changes"], indent=2))
display(HTML((run.root / "diagnosis/inspection.html").read_text()))


## Preserve artifacts

Download `milestone2_quartet_transfer_artifacts.zip` for review. Includes population
metadata, predictions, summaries, matched comparisons, reference metrics/hashes, preview,
protocol, revision/runtime and logs. Original reference weights remain in the input archive.


In [ ]:
archive = archive_quartet_transfer(run)
print("Transfer archive:", archive)
display(FileLink(str(archive)))
